In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA = Path("../data")
raw = pd.read_excel(DATA / "raw-micro(nano)plastics_database.xlsx")
model = pd.read_excel(DATA / "model_data (2).xlsx")

print("raw  :", raw.shape)
print("model:", model.shape)

raw  : (2119, 24)
model: (1902, 14)


#both tables


In [2]:
def overview(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "n_unique": df.nunique(),
        "example": df.iloc[0],
    })

overview(raw)

,dtype,missing,n_unique,example
Core,object,0,14,PS
Plastic - source,object,0,2,In-house
Shape,object,0,3,Beads
Concentration(μg/mL）,object,0,104,0.01
Diameter(nm),object,87,105,50
Size in water,object,1057,93,47 ± 1
Size in medium,object,1374,74,NaN
Zeta potential(water)(mV),object,960,95,-58.2
Zeta potential(other)(mV),object,1351,63,NaN
Surface charge,object,763,2,Negative


In [3]:
overview(model)

,dtype,missing,n_unique,example
Core,object,0,12,PS
Plastic source,object,0,2,In-house
Shape,object,0,2,Beads
Concentration(μg/mL),float64,0,83,0.01
Diameter(nm),float64,0,85,50.0
Surface modification,object,0,5,Carboxyl-modified
Cell line,object,0,70,Caco-2 : HT29-MTX-E12 (9:1)
Cell anatomical type,object,0,17,Epithelial
Cell source species,object,0,11,Human
Cell origin,object,0,2,Cell-line


- `model` has **no missing values**. `raw` has gaps, mainly in the particle characterisation columns (size in water/medium, zeta potential, surface charge) and 87 missing diameters.
- In `raw`, `Concentration` and `Diameter` are stored as **text**, because some cells hold other units (`μg/cm2`, `particles/mL`), ranges (`25000-75000`) or `mean ± sd` values.
- `model` keeps the 14 modelling columns and drops the characterisation columns and the paper info (`ref`, `Title`, `Journal`, and so on).

## 2. Duplicate rows

In [4]:
for name, df in [("raw", raw), ("model", model)]:
    print(f"{name}: {df.duplicated().sum()} exact duplicate row(s)")

raw: 1 exact duplicate row(s)
model: 1 exact duplicate row(s)


In [5]:
print("raw: exact duplicates")
display(raw[raw.duplicated(keep=False)])
print("model: exact duplicates")
display(model[model.duplicated(keep=False)])

raw: exact duplicates


,Core,Plastic - source,Shape,Concentration(μg/mL）,Diameter(nm),Size in water,Size in medium,Zeta potential(water)(mV),Zeta potential(other)(mV),Surface charge,Surface modification,Cell line,Cell-anatomical-type,Cell source species,Cell-origin,Cell-tissue-organ-origin,Assay type,Exposure time,Cell viability,ref,Title,Publication year,Journal,Author
89,PS,Commercial,Beads,12,60,63,415(DMEM),35.8,-3.7(DMEM),Positive,Amino-modified,RAW 264.7,Monocyte,Rat,Cell-line,Blood,MTS,16,27.1,4,Cationic Polystyrene Nanosphere Toxicity Depen...,2008.0,ACS Nano,"Tian Xia,†,# Michael Kovochich,†,# Monty Liong..."
145,PS,Commercial,Beads,12,60,63,415(DMEM),35.8,-3.7(DMEM),Positive,Amino-modified,RAW 264.7,Monocyte,Rat,Cell-line,Blood,MTS,16,27.1,4,Cationic Polystyrene Nanosphere Toxicity Depen...,2008.0,ACS Nano,"Tian Xia,†,# Michael Kovochich,†,# Monty Liong..."


model: exact duplicates


,Core,Plastic source,Shape,Concentration(μg/mL),Diameter(nm),Surface modification,Cell line,Cell anatomical type,Cell source species,Cell origin,Cell-tissue-organ-origin,Assay type,Exposure time,Cell viability
89,PS,Commercial,Beads,12.0,60.0,Amino-modified,Raw264.7,Monocyte,Rat,Cell-line,Blood,MTS,16,27.1
145,PS,Commercial,Beads,12.0,60.0,Amino-modified,Raw264.7,Monocyte,Rat,Cell-line,Blood,MTS,16,27.1


In [6]:
def normalise_text(df):
    """Lower-case text columns and remove all whitespace, so 'Human ' == 'human'."""
    out = df.copy()
    for c in out.columns:
        if not pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].astype(str).str.replace(r"\s+", "", regex=True).str.lower()
    return out

for name, df in [("raw", raw), ("model", model)]:
    print(f"{name}: {normalise_text(df).duplicated().sum()} duplicate row(s) after normalising text")

raw_n = normalise_text(raw)
new_dups = raw_n.duplicated(keep=False) & ~raw.duplicated(keep=False)
display(raw[new_dups])
print("Concentration values as stored:", raw.loc[new_dups, "Concentration(μg/mL）"].map(repr).tolist())

raw: 2 duplicate row(s) after normalising text
model: 1 duplicate row(s) after normalising text


,Core,Plastic - source,Shape,Concentration(μg/mL）,Diameter(nm),Size in water,Size in medium,Zeta potential(water)(mV),Zeta potential(other)(mV),Surface charge,Surface modification,Cell line,Cell-anatomical-type,Cell source species,Cell-origin,Cell-tissue-organ-origin,Assay type,Exposure time,Cell viability,ref,Title,Publication year,Journal,Author
1962,PS,Commercial,Beads,5000,4500,5009 ± 188,6777 ± 357(BME),-53.7±0.61,-22.5±1.99 (BME),Negative,Unmodified,Mussel-Hemocytes,Hemocyte,Mussel,Primary,Immune-system,MTT,24,69.4,71,Polystyrene nanoplastics and microplastics can...,2021.0,Scientific Reports,"Alberto Katsumiti, María Paula Losada‑Carrillo..."
1963,PS,Commercial,Beads,5000,4500,5009 ± 188,6777 ± 357(BME),-53.7±0.61,-22.5±1.99 (BME),Negative,Unmodified,Mussel-Hemocytes,Hemocyte,Mussel,Primary,Immune-system,MTT,24,69.4,71,Polystyrene nanoplastics and microplastics can...,2021.0,Scientific Reports,"Alberto Katsumiti, María Paula Losada‑Carrillo..."


Concentration values as stored: ['5000', "'5000'"]


### Attach the paper reference to the model data

In [13]:
meta = ["ref", "Title", "Publication year", "Journal"]
model_ref = model.copy()
model_ref[meta] = raw_r.loc[pairs.set_index("model_row").sort_index().raw_row, meta].to_numpy()

print("rows:", len(model_ref), "| missing ref:", model_ref["ref"].isna().sum(), "| papers:", model_ref["ref"].nunique())
out = DATA / "cleaned"
out.mkdir(exist_ok=True)
model_ref.to_csv(out / "model_data_with_ref.csv", index=False)
model_ref.head()

rows: 1902 | missing ref: 0 | papers: 71


,Core,Plastic source,Shape,Concentration(μg/mL),Diameter(nm),Surface modification,Cell line,Cell anatomical type,Cell source species,Cell origin,Cell-tissue-organ-origin,Assay type,Exposure time,Cell viability,ref,Title,Publication year,Journal
0,PS,In-house,Beads,0.01,50.0,Carboxyl-modified,Caco-2 : HT29-MTX-E12 (9:1),Epithelial,Human,Cell-line,Colon,WST-1,24,100.1,1,Multi-endpoint toxicological assessment of pol...,2019.0,Toxicology in Vitro
1,PS,In-house,Beads,0.10,50.0,Carboxyl-modified,Caco-2 : HT29-MTX-E12 (9:1),Epithelial,Human,Cell-line,Colon,WST-1,24,100.5,1,Multi-endpoint toxicological assessment of pol...,2019.0,Toxicology in Vitro
2,PS,In-house,Beads,1.00,50.0,Carboxyl-modified,Caco-2 : HT29-MTX-E12 (9:1),Epithelial,Human,Cell-line,Colon,WST-1,24,94.6,1,Multi-endpoint toxicological assessment of pol...,2019.0,Toxicology in Vitro
3,PS,In-house,Beads,5.00,50.0,Carboxyl-modified,Caco-2 : HT29-MTX-E12 (9:1),Epithelial,Human,Cell-line,Colon,WST-1,24,98.9,1,Multi-endpoint toxicological assessment of pol...,2019.0,Toxicology in Vitro
4,PS,In-house,Beads,10.00,50.0,Carboxyl-modified,Caco-2 : HT29-MTX-E12 (9:1),Epithelial,Human,Cell-line,Colon,WST-1,24,97.3,1,Multi-endpoint toxicological assessment of pol...,2019.0,Toxicology in Vitro
